In [ ]:
# Colab‑specific imports for displaying images
from google.colab.patches import cv2_imshow  # ✅ For OpenCV display in Colab
from IPython.display import display          # ✅ For PIL display in Colab

# Standard imports
import cv2
import numpy as np
from PIL import Image

# Path to your image
img_path = "document.jpg"

# 1️⃣ OpenCV: Load the image
img_cv = cv2.imread(img_path)

# ⛔️ Only for non-Colab (won’t work in notebooks)
# cv2.imshow("OpenCV Image", img_cv)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

# ✅ For Colab: use this to display OpenCV image inline
cv2_imshow(img_cv)

# 2️⃣ PIL (Pillow): Load the image
img_pil = Image.open(img_path)

# ⛔️ Only for non-Colab
# img_pil.show()

# ✅ For Colab: inline display using IPython
display(img_pil)

# 3️⃣ NumPy: Convert image to array and print shape
img_np = np.array(img_pil)
print("NumPy Image Shape:", img_np.shape)

In [7]:
import pandas as pd
import requests
from io import StringIO
import time

# ---------------------------
# CONFIGURATION (Corrected 11-digit IDs)
# ---------------------------
# Training: KDEN, KDFW (Inland) + KSFO (Coastal)
TRAIN_STATIONS = {
    "KDEN": "72565003017", "KDFW": "72259003927", "KSFO": "72494023234"
}

# Testing: KATL (Inland) + KBOS (Coastal)
TEST_STATIONS = {
    "KATL": "72219013874", "KBOS": "72509014739"
}

def fetch_and_clean(station_dict, start_year, end_year):
    all_data = []
    for name, s_id in station_dict.items():
        print(f"Fetching {name} ({s_id})...")
        
        url = "https://www.ncei.noaa.gov/access/services/data/v1"
        params = {
            "dataset": "global-hourly",
            "stations": s_id,
            "startDate": f"{start_year}-01-01T00:00:00",
            "endDate": f"{end_year}-12-31T23:59:59",
            "dataTypes": "TMP,DEW,VIS,WSPD",
            "format": "csv", 
            "units": "metric"
        }
        
        try:
            res = requests.get(url, params=params, timeout=30)
            res.raise_for_status()
            
            # Check if we actually got CSV data or just an empty response
            if len(res.text.strip()) < 100:
                print(f"  ⚠️ No data returned for {name}. Checking ID or dates...")
                continue
                
            df = pd.read_csv(StringIO(res.text))
            
            # NOAA flags are like '0120,1'. We take the part before the comma.
            def parse_noaa_col(col_name, divisor=1):
                if col_name not in df.columns: return pd.NA
                return df[col_name].astype(str).str.split(',').str[0].replace(['9999', '999.9', '99.99'], pd.NA).astype(float) / divisor

            df['temp'] = parse_noaa_col('TMP', 10)
            df['dew']  = parse_noaa_col('DEW', 10)
            df['vis']  = parse_noaa_col('VIS', 1)
            df['wind'] = parse_noaa_col('WSPD', 10)
            
            df['timestamp'] = pd.to_datetime(df['DATE'])
            df['airport'] = name
            
            # Drop rows where all weather data is missing
            clean_df = df.dropna(subset=['temp', 'dew', 'vis', 'wind'], how='all')
            all_data.append(clean_df[['timestamp', 'airport', 'temp', 'dew', 'vis', 'wind']])
            
            print(f"  ✅ Successfully retrieved {len(clean_df)} rows.")
            time.sleep(1) 
            
        except Exception as e:
            print(f"  ❌ Error fetching {name}: {e}")
            
    if not all_data:
        return pd.DataFrame(columns=['timestamp', 'airport', 'temp', 'dew', 'vis', 'wind'])
        
    return pd.concat(all_data).sort_values(['airport', 'timestamp'])

# --- RUN ---
train_df = fetch_and_clean(TRAIN_STATIONS, 2018, 2023)
test_df = fetch_and_clean(TEST_STATIONS, 2024, 2025)

Fetching KDEN (72565003017)...
  ❌ Error fetching KDEN: HTTPSConnectionPool(host='www.ncei.noaa.gov', port=443): Max retries exceeded with url: /access/services/data/v1?dataset=global-hourly&stations=72565003017&startDate=2018-01-01T00%3A00%3A00&endDate=2023-12-31T23%3A59%3A59&dataTypes=TMP%2CDEW%2CVIS%2CWSPD&format=csv&units=metric (Caused by NameResolutionError("HTTPSConnection(host='www.ncei.noaa.gov', port=443): Failed to resolve 'www.ncei.noaa.gov' ([Errno 11002] getaddrinfo failed)"))
Fetching KDFW (72259003927)...
  ❌ Error fetching KDFW: HTTPSConnectionPool(host='www.ncei.noaa.gov', port=443): Max retries exceeded with url: /access/services/data/v1?dataset=global-hourly&stations=72259003927&startDate=2018-01-01T00%3A00%3A00&endDate=2023-12-31T23%3A59%3A59&dataTypes=TMP%2CDEW%2CVIS%2CWSPD&format=csv&units=metric (Caused by NameResolutionError("HTTPSConnection(host='www.ncei.noaa.gov', port=443): Failed to resolve 'www.ncei.noaa.gov' ([Errno 11002] getaddrinfo failed)"))
Fetching